In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import joblib


In [2]:
df = pd.read_csv("/kaggle/input/datasets/ossamasamir/used-iphone-datasetprice/used_iphone_egypt_images_dataset.csv")
df.head()

,model,ram_gb,storage_gb,color,battery_health_pct,price_egp,warranty
0,iPhone 14,6,128,Blue,100.0,22600,Unknown
1,iPhone 11,4,128,Black,71.0,67500,Yes
2,iPhone 16,8,128,Blue,95.0,81000,No
3,iPhone 13,4,256,Midnight,90.0,63900,No
4,iPhone 12 Pro Max,6,256,Pacific Blue,75.0,27250,Unknown


###  Check the dataset structure

In [3]:
df.info

<bound method DataFrame.info of                    model  ram_gb  storage_gb         color  \
0              iPhone 14       6         128          Blue   
1              iPhone 11       4         128         Black   
2              iPhone 16       8         128          Blue   
3              iPhone 13       4         256      Midnight   
4      iPhone 12 Pro Max       6         256  Pacific Blue   
...                  ...     ...         ...           ...   
26419         iPhone 17e       8         512         White   
26420          iPhone 6s       2          32        Silver   
26421      iPhone 7 Plus       3         128        Silver   
26422          iPhone 17       8         256          Sage   
26423         iPhone Air      12         256    Light Gold   

       battery_health_pct  price_egp warranty  
0                   100.0      22600  Unknown  
1                    71.0      67500      Yes  
2                    95.0      81000       No  
3                    90.0      

### Check the missing rows

In [4]:
df.isnull().sum()

model                 0
ram_gb                0
storage_gb            0
color                 0
battery_health_pct    7
price_egp             0
warranty              0
dtype: int64

In [5]:
df[df["battery_health_pct"].isna()]
# This will show us the 7 rows with missing battery health.

,model,ram_gb,storage_gb,color,battery_health_pct,price_egp,warranty
8,iPhone 12 Pro Max,6,256,Pacific Blue,NaN,31750,No
9,iPhone 14 Pro Max,6,256,Black (Shade B),NaN,33000,No
10,iPhone 16 Pro,8,512,Natural Titanium,NaN,49450,No
11,iPhone 14 Pro Max,6,128,Deep Purple,NaN,31350,No
64,iPhone 17 Pro Max,12,256,Silver,NaN,72000,No
72,iPhone 17 Pro Max,12,256,Silver,NaN,72000,No
99,iPhone 16,8,128,Black,NaN,61740,No


In [6]:
df["battery_health_pct"] = df["battery_health_pct"].fillna(
    df["battery_health_pct"].median()
)

In [7]:
df.isnull().sum()

model                 0
ram_gb                0
storage_gb            0
color                 0
battery_health_pct    0
price_egp             0
warranty              0
dtype: int64

## Check categorical values
> Encoding is the next step.

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26424 entries, 0 to 26423
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   model               26424 non-null  object 
 1   ram_gb              26424 non-null  int64  
 2   storage_gb          26424 non-null  int64  
 3   color               26424 non-null  object 
 4   battery_health_pct  26424 non-null  float64
 5   price_egp           26424 non-null  int64  
 6   warranty            26424 non-null  object 
dtypes: float64(1), int64(3), object(3)
memory usage: 1.4+ MB


In [9]:
print(df['model'].unique())
print(df['color'].unique())
print(df['warranty'].unique())
# check their actual values so we choose the right encoding method.

['iPhone 14' 'iPhone 11' 'iPhone 16' 'iPhone 13' 'iPhone 12 Pro Max'
 'iPhone 14 Pro Max' 'iPhone 16 Pro' 'iPhone 16 Pro Max'
 'iPhone 15 Pro Max' 'iPhone 15 Pro' 'iPhone 15' 'iPhone 17 Pro Max'
 'iPhone 17' 'iPhone 15 Plus' 'iPhone 16 Air' 'iPhone 17 Pro'
 'iPhone 16 Plus' 'iPhone 12 Pro' 'iPhone 14 Pro' 'iPhone XR'
 'iPhone XS Max' 'iPhone 13 Pro' 'iPhone 12' 'iPhone 14 Plus' 'iPhone XS'
 'iPhone 13 mini' 'iPhone X' 'iPhone 13 Pro Max' 'iPhone 11 Pro Max'
 'iPhone 12 mini' 'iPhone 11 Pro' 'iPhone 17e' 'iPhone 6s Plus'
 'iPhone 6s' 'iPhone 8' 'iPhone 5s' 'iPhone 5c' 'iPhone 7 Plus' 'iPhone 7'
 'iPhone SE (2nd gen)' 'iPhone 5' 'iPhone 6' 'iPhone 8 Plus'
 'iPhone 6 Plus' 'iPhone Air' 'iPhone 16e']
['Blue' 'Black' 'Midnight' 'Pacific Blue' 'Black (Shade B)'
 'Natural Titanium' 'Deep Purple' 'Purple' 'Rose Gold' 'Black Titanium'
 'Starlight' 'Blue Titanium' 'Cosmic Orange' 'Gold' 'White' 'Teal'
 'Space Black' 'Silver' 'Titanium (color text truncated)' 'Deep Blue'
 'Ultramarine' 'Purple Fa

In [10]:
le_model = LabelEncoder()
le_color = LabelEncoder()
le_warranty = LabelEncoder()

df["model"] = le_model.fit_transform(df["model"])
df["color"] = le_color.fit_transform(df["color"])
df["warranty"] = le_warranty.fit_transform(df["warranty"])

In [11]:
df.head()

,model,ram_gb,storage_gb,color,battery_health_pct,price_egp,warranty
0,11,6,128,4,100.0,22600,1
1,0,4,128,1,71.0,67500,2
2,19,8,128,4,95.0,81000,0
3,7,4,256,19,90.0,63900,0
4,5,6,256,22,75.0,27250,1


In [12]:
X = df.drop("price_egp", axis=1)
y = df["price_egp"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (21139, 6)
X_test: (5285, 6)
y_train: (21139,)
y_test: (5285,)


In [15]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [16]:
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [17]:
y_pred = model.predict(X_test)

In [18]:
print(y_pred[:10])

[33120.8047619  35332.04310967  7586.575      13106.60436508
 31532.75       30012.44920635 60102.75        7362.66666667
 45010.94761905 30613.17907648]


In [19]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE: 1220.0984707833322
RMSE: 1820.2766627187282
R² Score: 0.993910475762802


In [20]:
joblib.dump(model, "iphone_price_model.pkl")
joblib.dump(le_model, "model_encoder.pkl")
joblib.dump(le_color, "color_encoder.pkl")
joblib.dump(le_warranty, "warranty_encoder.pkl")

['warranty_encoder.pkl']